# Phase 01 — Confirm PaddleOCR Setup on Kaggle

**Goal:** Record exact versions, models, configs, and train/eval/export entrypoints.  
**Runs on:** Kaggle only (enable GPU if available).  
**Does not:** train on NID data, read local PC files, or need `Docs/` mounted.

| In scope | Out of scope |
| --- | --- |
| `PP-OCRv5_server_det` + `PP-OCRv5_server_rec` | Preprocessing / parser / field extraction |
| Unified Bangla + English recognizer path | Separate Bengali OCR model |
| Pin packages + clone PaddleOCR | Accuracy claims from 2–4 NIDs |

**Output:** `/kaggle/working/VERSION_NOTES.md` → copy into repo as `Docs/VERSION_NOTES.md` after review.

### How to run

1. Enable **GPU** accelerator.
2. Run cells top → bottom through **Install** (section 2).
3. **Restart session** (Kaggle → Restart & clear outputs is fine).
4. Run **again from section 1** top → bottom. Section 2 skips install if packages are already present.

## 1. Environment snapshot

In [1]:
import sys
import os
import platform
import subprocess
from pathlib import Path

WORKDIR = Path('/kaggle/working')
INPUT_ROOT = Path('/kaggle/input')
OUTPUT_DIR = WORKDIR / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PYTHON_VERSION = sys.version.split()[0]
PLATFORM = platform.platform()

print('Python :', sys.version)
print('Platform:', PLATFORM)
print('WORKDIR :', WORKDIR)
print('INPUT   :', INPUT_ROOT, 'exists=', INPUT_ROOT.exists())

print('\n=== GPU / CUDA ===')
print('CUDA_VISIBLE_DEVICES:', os.environ.get('CUDA_VISIBLE_DEVICES', '(unset)'))
nvidia_ok = False
try:
    r = subprocess.run(['nvidia-smi'], capture_output=True, text=True, timeout=30)
    nvidia_ok = r.returncode == 0
    print(r.stdout if nvidia_ok else (r.stderr or 'nvidia-smi failed'))
except FileNotFoundError:
    print('nvidia-smi not found (CPU session).')

ACCELERATOR = 'GPU' if nvidia_ok else 'CPU'
print('Accelerator:', ACCELERATOR)

Python : 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.12.90+-x86_64-with-glibc2.35
WORKDIR : /kaggle/working
INPUT   : /kaggle/input exists= True

=== GPU / CUDA ===
CUDA_VISIBLE_DEVICES: (unset)
Fri Sep 18 15:00:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C 

## 2. Install packages (run once, then Restart)

Targeted for current Kaggle (Python 3.12 + modern NVIDIA driver / CUDA 12–13).

| Package | Pin |
| --- | --- |
| paddleocr | 3.3.0 |
| paddlex | 3.3.3 |
| paddlepaddle-gpu | 3.3.0 (cu130 → cu126 → CPU fallback) |

> After this cell finishes: **Restart session**, then continue from section 3.  
> Ignore unrelated pip warnings about `google-adk` / `ydata-profiling`.

In [2]:
import sys
import subprocess
import importlib.metadata as md


def pkg_ver(name: str) -> str:
    try:
        return md.version(name)
    except md.PackageNotFoundError:
        return 'NOT INSTALLED'


need_paddle = pkg_ver('paddlepaddle') == 'NOT INSTALLED'
need_ocr = pkg_ver('paddleocr') == 'NOT INSTALLED'
need_x = pkg_ver('paddlex') == 'NOT INSTALLED'

print('Before install:')
print('  paddlepaddle:', pkg_ver('paddlepaddle'))
print('  paddleocr   :', pkg_ver('paddleocr'))
print('  paddlex     :', pkg_ver('paddlex'))

if need_ocr or need_x:
    print('\\nInstalling paddleocr + paddlex ...')
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q',
         'paddleocr==3.3.0', 'paddlex==3.3.3'],
        check=True,
    )

if need_paddle:
    print('\\nInstalling paddlepaddle (GPU preferred) ...')
    candidates = [
        ['paddlepaddle-gpu==3.3.0', '-i', 'https://www.paddlepaddle.org.cn/packages/stable/cu130/'],
        ['paddlepaddle-gpu==3.3.0', '-i', 'https://www.paddlepaddle.org.cn/packages/stable/cu126/'],
        ['paddlepaddle==3.3.0', '-i', 'https://www.paddlepaddle.org.cn/packages/stable/cpu/'],
    ]
    installed = False
    for args in candidates:
        cmd = [sys.executable, '-m', 'pip', 'install', '-q', *args]
        print('Trying:', ' '.join(args))
        r = subprocess.run(cmd, capture_output=True, text=True)
        try:
            md.packages_distributions.cache_clear()
        except Exception:
            pass
        if r.returncode == 0 and pkg_ver('paddlepaddle') != 'NOT INSTALLED':
            print('OK:', pkg_ver('paddlepaddle'))
            installed = True
            break
        err = (r.stderr or r.stdout or '')[-800:]
        print('Failed:\\n', err)
    if not installed:
        raise RuntimeError('Could not install paddlepaddle. Check network / try CPU index manually.')
else:
    print('\\npaddlepaddle already present — skipping.')

print('\\nAfter install:')
print('  paddlepaddle:', pkg_ver('paddlepaddle'))
print('  paddleocr   :', pkg_ver('paddleocr'))
print('  paddlex     :', pkg_ver('paddlex'))
print('\\n>>> RESTART THE KAGGLE SESSION NOW, then run again from section 1 <<<')


Before install:
  paddlepaddle: NOT INSTALLED
  paddleocr   : NOT INSTALLED
  paddlex     : NOT INSTALLED
\nInstalling paddleocr + paddlex ...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.9/42.9 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.0/81.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.5/767.5 kB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.2/67.2 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 108.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 MB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.1/204.1 kB 14.7 MB

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
ydata-profiling 4.18.4 requires PyYAML<6.1,>=6.0.3, but you have pyyaml 6.0.2 which is incompatible.


\nInstalling paddlepaddle (GPU preferred) ...
Trying: paddlepaddle-gpu==3.3.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu130/
Failed:\n licts.
pylibcudf-cu12 26.2.1 requires cuda-python<13.0,>=12.9.2, but you have cuda-python 13.0.3 which is incompatible.
rmm-cu12 26.2.0 requires cuda-python<13.0,>=12.9.2, but you have cuda-python 13.0.3 which is incompatible.
cuml-cu12 26.2.0 requires cuda-python<13.0,>=12.9.2, but you have cuda-python 13.0.3 which is incompatible.
pylibraft-cu12 26.2.0 requires cuda-python<13.0,>=12.9.2, but you have cuda-python 13.0.3 which is incompatible.
cuvs-cu12 26.2.0 requires cuda-python<13.0,>=12.9.2, but you have cuda-python 13.0.3 which is incompatible.
cudf-cu12 26.2.1 requires cuda-python<13.0,>=12.9.2, but you have cuda-python 13.0.3 which is incompatible.
torch 2.10.0+cu128 requires cuda-bindings==12.9.4; platform_system == "Linux", but you have cuda-bindings 13.0.3 which is incompatible.

Trying: paddlepaddle-gpu==3.3.0 -i https://www.paddle

## 3. Verify installed versions

After restart, re-run section 1 first (redefines paths), then this cell.

In [3]:
import importlib.metadata as md

REFERENCE_VERSIONS = {
    'paddleocr': '3.3.0',
    'paddlepaddle': '3.3.0',  # Kaggle pin; older report used 3.2.0 on another machine
    'paddlex': '3.3.3',
}

versions = {}
for pkg in REFERENCE_VERSIONS:
    try:
        versions[pkg] = md.version(pkg)
    except md.PackageNotFoundError:
        versions[pkg] = 'NOT INSTALLED'

print('Installed vs target:')
for pkg, ref in REFERENCE_VERSIONS.items():
    print(f'  {pkg:12} installed={versions[pkg]:<16} target={ref}')

missing = [p for p, v in versions.items() if v == 'NOT INSTALLED']
if missing:
    raise RuntimeError(f'Missing packages {missing}. Re-run Install (section 2) and Restart.')

import paddle
print('\npaddle.__version__         :', paddle.__version__)
print('paddle compiled with CUDA :', paddle.is_compiled_with_cuda())
try:
    print('paddle.device             :', paddle.device.get_device())
except Exception as exc:
    print('paddle.device error       :', exc)

versions

Installed vs target:
  paddleocr    installed=3.3.0            target=3.3.0
  paddlepaddle installed=3.3.0            target=3.3.0
  paddlex      installed=3.3.3            target=3.3.3


/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)



paddle.__version__         : 3.3.0
paddle compiled with CUDA : False
paddle.device             : cpu


{'paddleocr': '3.3.0', 'paddlepaddle': '3.3.0', 'paddlex': '3.3.3'}

## 4. Clone / pin PaddleOCR under `/kaggle/working/`

Records the exact Git commit. Optional: set `PADDLEOCR_GIT_REF` to a release tag.

In [4]:
PADDLEOCR_DIR = WORKDIR / 'PaddleOCR'
PADDLEOCR_GIT_REF = os.environ.get('PADDLEOCR_GIT_REF', '').strip()

if not PADDLEOCR_DIR.exists():
    clone_cmd = ['git', 'clone', 'https://github.com/PaddlePaddle/PaddleOCR.git', str(PADDLEOCR_DIR)]
    if PADDLEOCR_GIT_REF:
        clone_cmd[2:2] = ['--branch', PADDLEOCR_GIT_REF, '--depth', '1']
    else:
        clone_cmd[2:2] = ['--depth', '1']
    subprocess.run(clone_cmd, check=True)
elif PADDLEOCR_GIT_REF:
    subprocess.run(['git', 'fetch', '--depth', '1', 'origin', 'tag', PADDLEOCR_GIT_REF],
                   cwd=PADDLEOCR_DIR, check=False)
    subprocess.run(['git', 'checkout', PADDLEOCR_GIT_REF], cwd=PADDLEOCR_DIR, check=False)

os.chdir(PADDLEOCR_DIR)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
try:
    describe = subprocess.check_output(
        ['git', 'describe', '--tags', '--always'], text=True, stderr=subprocess.DEVNULL
    ).strip()
except subprocess.CalledProcessError:
    describe = '(no tag)'

print('Repo     :', PADDLEOCR_DIR)
print('Ref      :', PADDLEOCR_GIT_REF or '(default shallow tip)')
print('Commit   :', commit)
print('Describe :', describe)

Cloning into '/kaggle/working/PaddleOCR'...


Repo     : /kaggle/working/PaddleOCR
Ref      : (default shallow tip)
Commit   : dab3fe35379033fdcb2d0e9572fac0b36c9a9ebf
Describe : dab3fe3


## 5. Confirm starting models

- Detection: `PP-OCRv5_server_det`
- Recognition: `PP-OCRv5_server_rec`

Do not switch to mobile variants.

In [6]:
DET_MODEL = 'PP-OCRv5_server_det'
REC_MODEL = 'PP-OCRv5_server_rec'
print('Detection :', DET_MODEL)
print('Recognition:', REC_MODEL)

candidate_roots = [
    Path.home() / '.paddlex' / 'official_models',
    Path('/root/.paddlex/official_models'),
    WORKDIR / '.paddlex' / 'official_models',
]
found_model_dirs = {}
print('\nCached official model dirs:')
for root in candidate_roots:
    if not root.exists():
        continue
    for name in (DET_MODEL, REC_MODEL):
        p = root / name
        print(('  FOUND ' if p.exists() else '  miss  '), p)
        if p.exists():
            found_model_dirs[name] = str(p)
if not found_model_dirs:
    print('  (none yet — first inference/train will download)')

Detection : PP-OCRv5_server_det
Recognition: PP-OCRv5_server_rec

Cached official model dirs:
  (none yet — first inference/train will download)


## 6. Find PP-OCRv5 training configs

Search both the cloned repo and the installed PaddleX package (PaddleOCR 3.x often trains via PaddleX).

In [7]:
def find_configs(roots, patterns):
    hits, seen = [], set()
    for root in roots:
        if root is None or not Path(root).exists():
            continue
        root = Path(root)
        for pat in patterns:
            for p in root.rglob(pat):
                key = str(p.resolve())
                if key not in seen:
                    seen.add(key)
                    hits.append(p)
    return hits

repo_config_root = PADDLEOCR_DIR / 'configs'
paddlex_roots = []
try:
    import paddlex
    paddlex_dir = Path(paddlex.__file__).resolve().parent
    paddlex_roots.append(paddlex_dir)
    print('paddlex dir:', paddlex_dir)
except Exception as exc:
    print('paddlex import failed:', exc)

rec_patterns = [
    '*PP-OCRv5*server*rec*.yml', '*PP-OCRv5*server*rec*.yaml',
    '**/text_recognition/**/PP-OCRv5_server_rec.yaml',
]
det_patterns = [
    '*PP-OCRv5*server*det*.yml', '*PP-OCRv5*server*det*.yaml',
    '**/text_detection/**/PP-OCRv5_server_det.yaml',
]

search_roots = [repo_config_root, *paddlex_roots]
rec_candidates = find_configs(search_roots, rec_patterns)
det_candidates = find_configs(search_roots, det_patterns)

print('\nRecognition configs:')
for p in rec_candidates:
    print(' -', p)
print('\nDetection configs:')
for p in det_candidates:
    print(' -', p)

if not rec_candidates or not det_candidates:
    raise FileNotFoundError('Missing det/rec PP-OCRv5 server configs — check clone / paddlex install.')

paddlex import failed: No module named 'langchain.docstore'

Recognition configs:
 - /kaggle/working/PaddleOCR/configs/rec/PP-OCRv5/PP-OCRv5_server_rec.yml

Detection configs:
 - /kaggle/working/PaddleOCR/configs/det/PP-OCRv5/PP-OCRv5_server_det.yml


## 7. Inspect recognition + detection configs

In [8]:
REC_CONFIG = rec_candidates[0]
DET_CONFIG = det_candidates[0]
print('REC_CONFIG:', REC_CONFIG)
print('DET_CONFIG:', DET_CONFIG)

rec_text = REC_CONFIG.read_text(encoding='utf-8')
det_text = DET_CONFIG.read_text(encoding='utf-8')

rec_keywords = [
    'pretrained_model', 'pretrain_weight_path', 'character_dict_path',
    'char_dict_path', 'dict_path', 'use_space_char', 'save_model_dir',
    'label_file_list', 'data_dir', 'dataset_dir',
]
det_keywords = [
    'pretrained_model', 'pretrain_weight_path', 'save_model_dir',
    'label_file_list', 'data_dir', 'dataset_dir',
]

print('\n--- Recognition relevant lines ---')
for i, line in enumerate(rec_text.splitlines(), 1):
    if any(k in line for k in rec_keywords):
        print(f'{i:4}: {line}')

print('\n--- Detection relevant lines ---')
for i, line in enumerate(det_text.splitlines(), 1):
    if any(k in line for k in det_keywords):
        print(f'{i:4}: {line}')

has_char_dict = any(k in rec_text for k in ('character_dict_path', 'char_dict_path', 'dict_path'))
has_space_flag = 'use_space_char' in rec_text
has_pretrain = any(k in rec_text for k in ('pretrained_model', 'pretrain_weight_path'))
print('\ndict field:', has_char_dict, '| use_space_char:', has_space_flag, '| pretrain field:', has_pretrain)

REC_CONFIG: /kaggle/working/PaddleOCR/configs/rec/PP-OCRv5/PP-OCRv5_server_rec.yml
DET_CONFIG: /kaggle/working/PaddleOCR/configs/det/PP-OCRv5/PP-OCRv5_server_det.yml

--- Recognition relevant lines ---
   8:   save_model_dir: ./output/PP-OCRv5_server_rec
  13:   pretrained_model: 
  18:   character_dict_path: ./ppocr/utils/dict/ppocrv5_dict.txt
  21:   use_space_char: true
  81:     data_dir: ./train_data/
  83:     label_file_list:
 114:     data_dir: ./train_data
 115:     label_file_list:

--- Detection relevant lines ---
   8:   save_model_dir: ./output/PP-OCRv5_server_det
  15:   pretrained_model: https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PPHGNetV2_B4_ocr_det.pdparams
  73:     data_dir: ./train_data/icdar2015/text_localization/
  74:     label_file_list:
 141:     data_dir: ./train_data/icdar2015/text_localization/
 142:     label_file_list:

dict field: True | use_space_char: True | pretrain field: True


## 8. Train / eval / export entrypoints

Confirm classic `tools/*.py` and/or PaddleX `main.py` / CLI on this install.

In [9]:
classic_entrypoints = {
    'train': PADDLEOCR_DIR / 'tools' / 'train.py',
    'eval': PADDLEOCR_DIR / 'tools' / 'eval.py',
    'export': PADDLEOCR_DIR / 'tools' / 'export_model.py',
}

print('Classic tools:')
for name, ep in classic_entrypoints.items():
    try:
        shown = ep.relative_to(PADDLEOCR_DIR)
    except ValueError:
        shown = ep
    print(f'  {name:7}: {shown} ->', 'FOUND' if ep.exists() else 'MISSING')

paddlex_mains = []
for root in [PADDLEOCR_DIR, *paddlex_roots]:
    root = Path(root)
    if not root.exists():
        continue
    for p in list(root.glob('main.py')) + list(root.glob('*/main.py')) + list(root.glob('*/*/main.py')):
        paddlex_mains.append(p.resolve())
paddlex_mains = list(dict.fromkeys(paddlex_mains))

print('\nmain.py candidates:')
for p in paddlex_mains[:10]:
    print(' ', p)
if not paddlex_mains:
    print('  (none found)')

print('\nCLI:')
for mod in ('paddleocr', 'paddlex'):
    try:
        r = subprocess.run([sys.executable, '-m', mod, '-h'], capture_output=True, text=True, timeout=60)
        print(f'  python -m {mod}: exit={r.returncode}')
        for line in (r.stdout or r.stderr or '').strip().splitlines()[:4]:
            print('   ', line)
    except Exception as exc:
        print(f'  python -m {mod}: {exc}')

TRAIN_ENTRY_CLASSIC = classic_entrypoints['train'].exists()
EVAL_ENTRY_CLASSIC = classic_entrypoints['eval'].exists()
EXPORT_ENTRY_CLASSIC = classic_entrypoints['export'].exists()
PADDLEX_MAIN = paddlex_mains[0] if paddlex_mains else None


def rel_or_abs(path: Path) -> str:
    try:
        return str(path.relative_to(PADDLEOCR_DIR))
    except ValueError:
        return str(path)

rec_rel = rel_or_abs(REC_CONFIG)
det_rel = rel_or_abs(DET_CONFIG)

print('\n=== Command templates (confirm keys before Phase 06/07) ===')
if TRAIN_ENTRY_CLASSIC:
    print(f'python tools/train.py -c {rec_rel} -o Global.pretrained_model=<REC_PRETRAINED_WEIGHTS>')
    print(f'python tools/eval.py -c {rec_rel} -o Global.pretrained_model=<REC_CHECKPOINT>')
    print(f'python tools/export_model.py -c {rec_rel} -o Global.pretrained_model=<REC_CHECKPOINT> Global.save_inference_dir=<REC_EXPORT_DIR>')
print(f'python main.py -c {rec_rel} -o Global.mode=train -o Global.dataset_dir=/kaggle/input/<rec-dataset> -o Train.pretrain_weight_path=<REC_PRETRAINED_WEIGHTS>')
print(f'python main.py -c {rec_rel} -o Global.mode=evaluate')
print(f'python main.py -c {rec_rel} -o Global.mode=export')
print(f'python main.py -c {det_rel} -o Global.mode=train -o Global.dataset_dir=/kaggle/input/<det-dataset> -o Train.pretrain_weight_path=<DET_PRETRAINED_WEIGHTS>')

Classic tools:
  train  : tools/train.py -> FOUND
  eval   : tools/eval.py -> FOUND
  export : tools/export_model.py -> FOUND

main.py candidates:
  (none found)

CLI:
  python -m paddleocr: exit=1
    Traceback (most recent call last):
      File "<frozen runpy>", line 189, in _run_module_as_main
      File "<frozen runpy>", line 148, in _get_module_details
      File "<frozen runpy>", line 112, in _get_module_details
  python -m paddlex: exit=1
    Traceback (most recent call last):
      File "<frozen runpy>", line 189, in _run_module_as_main
      File "<frozen runpy>", line 148, in _get_module_details
      File "<frozen runpy>", line 112, in _get_module_details

=== Command templates (confirm keys before Phase 06/07) ===
python tools/train.py -c configs/rec/PP-OCRv5/PP-OCRv5_server_rec.yml -o Global.pretrained_model=<REC_PRETRAINED_WEIGHTS>
python tools/eval.py -c configs/rec/PP-OCRv5/PP-OCRv5_server_rec.yml -o Global.pretrained_model=<REC_CHECKPOINT>
python tools/export_model.py

## 9. Bangla + English dictionary mechanism

Phase 01 only checks that a custom dict path exists in config and UTF-8 Bangla round-trips. Final charset = Phase 05.

In [10]:
TEST_DICT = WORKDIR / 'bangla_english_test_dict.txt'
test_chars = [
    'A', 'B', 'C', 'M', 'D', '0', '1', '2',
    'অ', 'আ', 'ই', 'ক', 'খ', 'গ',
    'া', 'ি', 'ী', 'ু', 'ূ', '্', 'ং', 'ঃ', 'ঁ',
    '.', '-', '/', ' ',
]
TEST_DICT.write_text('\n'.join(test_chars) + '\n', encoding='utf-8')
roundtrip = TEST_DICT.read_text(encoding='utf-8')
bangla_ok = all(ch in roundtrip for ch in ('অ', 'ক', '্', 'ঁ'))
print(roundtrip)
print('dict field present :', has_char_dict)
print('use_space_char     :', has_space_flag)
print('UTF-8 Bangla test  :', 'PASS' if bangla_ok else 'FAIL')
print('file:', TEST_DICT)

A
B
C
M
D
0
1
2
অ
আ
ই
ক
খ
গ
া
ি
ী
ু
ূ
্
ং
ঃ
ঁ
.
-
/
 

dict field present : True
use_space_char     : True
UTF-8 Bangla test  : PASS
file: /kaggle/working/bangla_english_test_dict.txt


## 10. Write `VERSION_NOTES.md`

Saved under `/kaggle/working/`. Download and copy into the GitHub repo as `Docs/VERSION_NOTES.md`.

In [11]:
VERSION_NOTES = WORKDIR / 'VERSION_NOTES.md'
det_model_path = found_model_dirs.get(DET_MODEL, 'Not cached yet')
rec_model_path = found_model_dirs.get(REC_MODEL, 'Not cached yet')

classic_train = (
    f'python tools/train.py -c {rec_rel} -o Global.pretrained_model=<REC_PRETRAINED_WEIGHTS>'
    if TRAIN_ENTRY_CLASSIC else 'classic tools/train.py not found'
)
paddlex_train = (
    f'python main.py -c {rec_rel} -o Global.mode=train '
    f'-o Global.dataset_dir=/kaggle/input/<rec-dataset> '
    f'-o Train.pretrain_weight_path=<REC_PRETRAINED_WEIGHTS>'
)

notes = f'''# Environment & PaddleOCR Version Notes

> Generated by `notebooks/01_phase01_environment.ipynb` on Kaggle.
> Copy into the repo as `Docs/VERSION_NOTES.md` after review.

## Python

Version: {PYTHON_VERSION}

Platform: {PLATFORM}

Accelerator: {ACCELERATOR}

## PaddlePaddle

Version: {versions.get('paddlepaddle')}

Compiled with CUDA: {paddle.is_compiled_with_cuda()}

## PaddleOCR

Version: {versions.get('paddleocr')}

Git commit: `{commit}`

Describe / tag: `{describe}`

Requested ref: `{PADDLEOCR_GIT_REF or 'default shallow tip'}`

Clone path: `{PADDLEOCR_DIR}`

## PaddleX

Version: {versions.get('paddlex')}

## Target Models

Detection:
{DET_MODEL}

Recognition:
{REC_MODEL}

Cached detection model path: `{det_model_path}`

Cached recognition model path: `{rec_model_path}`

## Training Entry Points

Classic tools/train.py: {TRAIN_ENTRY_CLASSIC}

Classic tools/eval.py: {EVAL_ENTRY_CLASSIC}

Classic tools/export_model.py: {EXPORT_ENTRY_CLASSIC}

PaddleX-style main.py: `{PADDLEX_MAIN or 'not found'}`

## Detection Config

`{det_rel}`

Absolute: `{DET_CONFIG}`

## Recognition Config

`{rec_rel}`

Absolute: `{REC_CONFIG}`

## Bangla + English Dictionary Behaviour

- Unified Bangla + English path in this recognizer (no second Bengali model).
- Config dict/charset field present: {has_char_dict}
- use_space_char present: {has_space_flag}
- UTF-8 Bangla round-trip: {'PASS' if bangla_ok else 'FAIL'}
- Final NID charset built in Phase 05.

## Train Command

```bash
{classic_train}
```

```bash
{paddlex_train}
```

## Evaluation Command

```bash
python tools/eval.py -c {rec_rel} -o Global.pretrained_model=<REC_CHECKPOINT>
# or
python main.py -c {rec_rel} -o Global.mode=evaluate
```

## Export Command

```bash
python tools/export_model.py -c {rec_rel} -o Global.pretrained_model=<REC_CHECKPOINT> Global.save_inference_dir=<REC_EXPORT_DIR>
# or
python main.py -c {rec_rel} -o Global.mode=export
```

## Kaggle Paths

- `/kaggle/input/<private-dataset-name>/` — NID data (Phase 02+)
- `/kaggle/working/PaddleOCR/` — pinned source
- `/kaggle/working/output/` — checkpoints / exports (later)

## Phase 01 Checklist

- [x] Kaggle environment recorded
- [x] Packages installed and verified
- [x] Models confirmed (server det + rec)
- [x] PaddleOCR commit recorded
- [x] Configs located
- [x] Train/eval/export entrypoints located
- [x] Bangla Unicode + dict mechanism checked
'''

VERSION_NOTES.write_text(notes, encoding='utf-8')
print(notes)
print('\nSaved:', VERSION_NOTES)

# Environment & PaddleOCR Version Notes

> Generated by `notebooks/01_phase01_environment.ipynb` on Kaggle.
> Copy into the repo as `Docs/VERSION_NOTES.md` after review.

## Python

Version: 3.12.13

Platform: Linux-6.12.90+-x86_64-with-glibc2.35

Accelerator: GPU

## PaddlePaddle

Version: 3.3.0

Compiled with CUDA: False

## PaddleOCR

Version: 3.3.0

Git commit: `dab3fe35379033fdcb2d0e9572fac0b36c9a9ebf`

Describe / tag: `dab3fe3`

Requested ref: `default shallow tip`

Clone path: `/kaggle/working/PaddleOCR`

## PaddleX

Version: 3.3.3

## Target Models

Detection:
PP-OCRv5_server_det

Recognition:
PP-OCRv5_server_rec

Cached detection model path: `Not cached yet`

Cached recognition model path: `Not cached yet`

## Training Entry Points

Classic tools/train.py: True

Classic tools/eval.py: True

Classic tools/export_model.py: True

PaddleX-style main.py: `not found`

## Detection Config

`configs/det/PP-OCRv5/PP-OCRv5_server_det.yml`

Absolute: `/kaggle/working/PaddleOCR/configs/de

## Phase 01 done when

You can answer:

1. Which PaddleOCR / PaddlePaddle / PaddleX versions are on this Kaggle session?
2. Which det/rec models are we starting from?
3. Where are the exact configs?
4. Which train / evaluate / export entrypoints exist?
5. How does the custom dictionary enter the recognizer config?
6. Does Bangla Unicode work in this environment?

Next: **Phase 02** — run stock models on 2–4 NIDs (private Kaggle Dataset) and save baseline under `/kaggle/working/experiments/phase02_baseline/`.